# MJX 08 — Playground PPO: PointMass reach

### Lab Description

A second MuJoCo Playground control task: **`PointMass`** (DM Control Suite). A 2-DoF
point mass on a plane must drive itself to a target. It has a **dense** reward
(higher the closer it gets), so PPO learns it cleanly in a couple of minutes on
the AMD GPU — a good contrast to the cartpole and a warm-up for the harder Panda
arm in MJX 09.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Train PPO on a second Playground task on the AMD GPU
- Reuse the ROCm-stable settings from MJX 07 (`num_envs=512`, `impl="jax"`, highest matmul precision, lower learning rate)
- Read the learning curve and render the trained policy

### Import libraries and apply the compatibility shim

Same setup as MJX 07: the `device_put_replicated` shim for brax-on-jax-rocm and the highest matmul precision for stable training.

In [ ]:
import os, functools
os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jp
# Brax 0.14 calls jax.device_put_replicated, removed in the jax 0.10 that ships
# with jax-rocm. Re-add a single-device version.
if not hasattr(jax, "device_put_replicated"):
    def _dpr(x, devices=None):
        n = len(devices) if devices is not None else jax.local_device_count()
        return jax.tree_util.tree_map(
            lambda a: jax.device_put(jp.broadcast_to(jp.asarray(a)[None], (n,) + jp.asarray(a).shape)), x)
    jax.device_put_replicated = _dpr
jax.config.update("jax_default_matmul_precision", "highest")

import numpy as np
import matplotlib.pyplot as plt
import imageio
from mujoco_playground import registry, wrapper
from mujoco_playground.config import dm_control_suite_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from IPython.display import Video

print("JAX devices:", jax.devices())

### Load PointMass and build the PPO config

We load the task with pure-JAX physics and start from Playground's tuned PPO config, then apply the same ROCm-stable overrides as MJX 07. Note `num_envs=512`: on this gfx1151 APU, 1024+ environments either crash (memory-aperture fault) or silently collapse to zero reward, so 512 is the largest batch that trains healthily.

In [ ]:
ENV_NAME = "PointMass"
env = registry.load(ENV_NAME, config_overrides={"impl": "jax"})

cfg = dm_control_suite_params.brax_ppo_config(ENV_NAME)
ppo_params = cfg.to_dict()
net_cfg = ppo_params.pop("network_factory", None)
ppo_params.update(
    # Same step-count note as MJX 07: brax rounds num_timesteps up to whole eval
    # intervals (~1.23M env steps each), so we set it to exactly (num_evals-1) of
    # them; the log ends at 5 * 1,228,800 = 6,144,000.
    num_evals=6,
    num_timesteps=6_144_000,
    num_envs=512,        # 1024+ is numerically unstable here (NaN / reward collapse); 512 is the safe ceiling
    batch_size=512,
    num_minibatches=8,
    learning_rate=3e-4,  # below the 1e-3 default, which diverges at this batch size on the APU
)
print({k: ppo_params[k] for k in ["num_timesteps", "num_envs", "batch_size", "episode_length"]})

### Train

Run PPO on the GPU. PointMass has a dense reward, so the eval reward should climb quickly and smoothly toward ~900 (the point mass parks on the target).

In [ ]:
progress = []
def progress_fn(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    progress.append((int(step), r))
    print(f"step {int(step):>9}  eval reward {r:8.1f}")

train_fn = functools.partial(ppo.train, **ppo_params)
if net_cfg:
    train_fn = functools.partial(train_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))

make_inference_fn, params, _ = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    seed=0,
)
print(f"training done — final eval reward {progress[-1][1]:.1f}")

### Plot the learning curve

In [ ]:
steps, rewards = zip(*progress)
plt.figure(figsize=(7, 3))
plt.plot(steps, rewards, marker="o")
plt.xlabel("environment steps"); plt.ylabel("eval episode reward")
plt.title(f"PPO learning curve ({ENV_NAME})"); plt.grid(True); plt.show()

### Render the trained policy from several starting points

The mass (red) starts at a random spot each episode and must drive to the target. To show the policy generalises, we roll out **four episodes from different random starts** and tile them in a 2x2 grid — every panel should converge on the target.

In [ ]:
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

def run_episode(seed, n=180):  # 180 steps @ 30 fps = 6 s
    state = reset(jax.random.PRNGKey(seed))
    rng = jax.random.PRNGKey(seed + 1000)
    traj = [state]
    for _ in range(n):
        rng, k = jax.random.split(rng)
        action, _ = inference(state.obs, k)
        state = step(state, action)
        traj.append(state)
    return np.asarray(env.render(traj, height=160, width=160))

# Four episodes from different random starts, tiled 2x2 to show the policy
# succeed from many starting configurations at once.
clips = [run_episode(seed) for seed in (1, 2, 3, 4)]
T = min(c.shape[0] for c in clips)
clips = [c[:T] for c in clips]
vsep = np.full((clips[0].shape[1], 4, 3), 255, np.uint8)   # vertical white divider
def tile(t):
    top = np.concatenate([clips[0][t], vsep, clips[1][t]], axis=1)
    bot = np.concatenate([clips[2][t], vsep, clips[3][t]], axis=1)
    hsep = np.full((4, top.shape[1], 3), 255, np.uint8)    # horizontal white divider
    return np.concatenate([top, hsep, bot], axis=0)
grid = np.stack([tile(t) for t in range(T)])

out = "output/videos/mjx08_point_mass.mp4"
imageio.mimsave(out, list(grid), fps=30)
print("saved", grid.shape[0], "frames of a 2x2 multi-start grid ->", out)

### Watch the result

In [ ]:
Video(url="output/videos/mjx08_point_mass.mp4")

## Conclusions

PPO solved a second Playground task with the same ROCm-stable recipe as the cartpole — only the task changed. You also saw the practical batch ceiling on this APU (`num_envs=512`): larger batches are numerically unhealthy here, so we keep training within that limit. MJX 09 takes the final step to a real robot arm, the Franka Panda pick-and-place.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT